# 11. AIND Units NWB Export

Run [aind-units-nwb](https://github.com/AllenNeuralDynamics/aind-units-nwb/tree/main/code) to append the curated sorting + waveforms to the NWB file produced by `10_aind_ecephys_nwb.ipynb`.

The capsule reads:
- a base `.nwb` (or `.nwb.zarr`) from `data/`
- the dispatcher's `job_*.json` (raw recording recipes)
- the collected `postprocessed/`, `curated/`, `spikesorted/` directories from notebook 7

and writes a copy of each base NWB into `results/` with a `units` table populated from the curated sorting (UUIDs, depth, amplitude, max channel) and per-unit `waveform_mean` / `waveform_sd`.

## Imports and deps

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "neuroconv", "pynwb", "hdmf-zarr", "aind-nwb-utils",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Checked 4 packages in 13ms


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'neuroconv', 'pynwb', 'hdmf-zarr', 'aind-nwb-utils'], returncode=0)

## Clone the capsule and seed `data/`

We need:
- the base NWB file from notebook 10 (`output/10_nwb_results/*.nwb`)
- the layout from notebook 7 (`output/07_collected_results/{postprocessed,curated,spikesorted}/...`)
- the dispatch `job_*.json`

We also patch a neuroconv API rename (`add_electrodes_info_to_nwbfile` → `add_electrodes_to_nwbfile`) the same way as notebook 10.

In [3]:
uw_repo = Path("/tmp/aind-units-nwb")
if not uw_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-units-nwb.git",
            str(uw_repo),
        ],
        check=True,
    )

data_dir = uw_repo / "data"
results_dir = uw_repo / "results"
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

for stale in list(data_dir.iterdir()) + list(results_dir.iterdir()):
    shutil.rmtree(stale) if stale.is_dir() else stale.unlink()

nwb_src = Path.cwd() / "output/10_nwb_results"
collected = Path.cwd() / "output/07_collected_results"
output/01_dispatch_results = Path.cwd() / "output/01_dispatch_results"
for src in (nwb_src, collected, output/01_dispatch_results):
    assert src.exists(), f"{src} missing - run earlier notebooks first."

for nwb in nwb_src.glob("*.nwb"):
    shutil.copy2(nwb, data_dir / nwb.name)

for entry in collected.iterdir():
    if entry.name in {"postprocessed", "curated", "spikesorted"}:
        shutil.copytree(entry, data_dir / entry.name)

for entry in output/01_dispatch_results.iterdir():
    if entry.name.startswith("job_"):
        shutil.copy2(entry, data_dir / entry.name)

# The capsule asserts exactly one ecephys_* folder is present in data/.
session_folder = data_dir / "ecephys_toy"
session_folder.mkdir()
(session_folder / "subject.json").write_text(json.dumps({"subject_id": "000000"}))

print("Seeded data dir:")
for p in sorted(data_dir.iterdir()):
    print(" ", p.name)

# Patch utils.py for neuroconv API drift:
# - `add_electrodes_info_to_nwbfile` -> `add_electrodes_to_nwbfile`
# - `add_units_table_to_nwbfile`     -> `_add_units_table_to_nwbfile` (now private)
utils_py = uw_repo / "code" / "utils.py"
src = utils_py.read_text()
patched = src.replace(
    "add_electrodes_info_to_nwbfile",
    "add_electrodes_to_nwbfile",
).replace(
    "add_units_table_to_nwbfile",
    "_add_units_table_to_nwbfile",
)
if patched != src:
    utils_py.write_text(patched)
    print("Patched utils.py for neuroconv API drift")

Cloning into '/tmp/aind-units-nwb'...


Seeded data dir:
  curated
  ecephys_toy
  job_0.json
  postprocessed
  session0_block0_recording1.nwb
  spikesorted
Patched utils.py for neuroconv API drift


## Run the units-NWB capsule

In [4]:
argv = ["python", "-u", "run_capsule.py"]
print("Running:", " ".join(argv))
result = subprocess.run(
    argv,
    cwd=uw_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"units-nwb run failed with code {result.returncode}")

Running: python -u run_capsule.py




NWB EXPORT UNITS
Running NWB conversion with the following parameters:
Stub test: False
Stub units: 10
NWB backend: hdf5
Found 1 JSON job files
Found 1 processed recordings
Number of NWB files to write: 1
Number of streams to write for each file: 1
	Adding 10 units from block0_None_recording1
Added 1 streams
Writing time: 0.05s
Done writing ../results/session0_block0_recording1.nwb
NWB EXPORT UNITS time: 1.32s



## Copy outputs next to the notebook

In [5]:
local_results_dir = Path.cwd() / "output/11_units_nwb_results"
local_results_dir.parent.mkdir(parents=True, exist_ok=True)
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
shutil.copytree(results_dir, local_results_dir)

for p in sorted(local_results_dir.rglob("*")):
    rel = p.relative_to(local_results_dir)
    kind = "dir" if p.is_dir() else f"{p.stat().st_size} bytes"
    print(f"  {rel}  ({kind})")

  session0_block0_recording1.nwb  (82341608 bytes)


## Inspect the units table

In [6]:
from pynwb import NWBHDF5IO

nwb_files = sorted(local_results_dir.glob("*.nwb"))
print("NWB files:", [p.name for p in nwb_files])

if nwb_files:
    with NWBHDF5IO(str(nwb_files[0]), "r") as io:
        nwb = io.read()
        units = nwb.units
        if units is None:
            print("No units table written.")
        else:
            print("units columns:", list(units.colnames))
            print("num_units:    ", len(units.id))
            df = units.to_dataframe().reset_index()
            keep = [c for c in ["id", "unit_name", "ks_unit_id", "device_name", "amplitude", "depth", "extremum_channel_index"] if c in df.columns]
            print()
            print(df[keep].to_string(index=False))

NWB files: ['session0_block0_recording1.nwb']
units columns: ['spike_times', 'electrodes', 'waveform_mean', 'waveform_sd', 'unit_name', 'spread', 'peak_before_width', 'velocity_above', 'isi_violations_count', 'decoder_probability', 'isolation_distance', 'num_positive_peaks', 'depth', 'nn_miss_rate', 'peak_after_width', 'amplitude', 'rp_violations', 'trough_half_width', 'sliding_rp_violation', 'extremum_channel_index', 'sync_spike_2', 'velocity_below', 'peak_after_to_trough_ratio', 'decoder_label', 'amplitude_median', 'firing_rate', 'nn_hit_rate', 'recovery_slope', 'sync_spike_4', 'isi_violations_ratio', 'drift_ptp', 'default_qc', 'rp_contamination', 'ks_unit_id', 'firing_range', 'd_prime', 'shank', 'exp_decay', 'snr', 'main_to_next_extremum_duration', 'trough_width', 'drift_std', 'original_cluster_id', 'amplitude_cv_median', 'repolarization_slope', 'num_negative_peaks', 'sync_spike_8', 'waveform_baseline_flatness', 'estimated_z', 'num_spikes', 'silhouette', 'amplitude_cutoff', 'device_